In [1]:
import sys 
print(sys.executable)
import torch
print(torch)

/home/vasco/nguyehmk/miniconda3/envs/llm/bin/python
<module 'torch' from '/home/vasco/nguyehmk/miniconda3/envs/llm/lib/python3.10/site-packages/torch/__init__.py'>


In [3]:
import warnings
import tokenizers
from transformers import AutoModelForCausalLM, AutoTokenizer,AutoConfig
from datasets import load_dataset

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import BitsAndBytesConfig

import evaluate
from transformers import TrainingArguments, Trainer

# !conda env export -n llm > environment.yml
# print("Env exported")


In [4]:
import psutil
import platform

print("===== SYSTEM INFO =====")
print("OS:", platform.system(), platform.release())
print("Processor:", platform.processor())
print("CPU cores:", psutil.cpu_count())

ram = psutil.virtual_memory()
print("RAM:", round(ram.total / (1024**3), 2), "GB")

print("\n===== GPU INFO =====")
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2), "GB")

===== SYSTEM INFO =====
OS: Linux 6.1.0-45-amd64
Processor: 
CPU cores: 32
RAM: 251.67 GB

===== GPU INFO =====
CUDA available: True
GPU: NVIDIA GeForce GTX 1080 Ti
VRAM: 10.91 GB


In [5]:
# Load model directly
torch.cuda.empty_cache()

# Check device availability
device = torch.device("cuda")
print("Model device:", device)

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-Coder-3B")
tokenizer.pad_token = tokenizer.eos_token
print("Load tokenizer completed")


# model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-Coder-3B", device_map="auto", torch_dtype="auto").to(device)

# print("Load model completed")



Model device: cuda


/home/vasco/nguyehmk/miniconda3/envs/llm/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Load tokenizer completed


In [6]:
# Code block for question - answering

messages = [
    {"role": "user", "content": "Write a python function for calculating the factorial of an integer."},
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(device)

outputs = model.generate(**inputs, max_new_tokens=500)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

NameError: name 'model' is not defined

In [7]:

ds = load_dataset("openai/openai_humaneval", split = "test")

In [8]:
#Structure of dataset dict
print(ds)

Dataset({
    features: ['task_id', 'prompt', 'canonical_solution', 'test', 'entry_point'],
    num_rows: 164
})


In [9]:
# Test pre process (tokenizer)
tokenizer(ds[0]['prompt'])

{'input_ids': [1499, 19496, 1159, 1759, 1406, 750, 702, 12704, 22801, 47207, 25, 1759, 95381, 1125, 12171, 25, 2224, 8, 1464, 1807, 510, 262, 4210, 4248, 421, 304, 2661, 1140, 315, 5109, 11, 525, 894, 1378, 5109, 12128, 311, 1817, 1008, 1091, 198, 262, 2661, 12171, 624, 262, 12109, 702, 12704, 22801, 2561, 16, 13, 15, 11, 220, 17, 13, 15, 11, 220, 18, 13, 15, 1125, 220, 15, 13, 20, 340, 262, 3557, 198, 262, 12109, 702, 12704, 22801, 2561, 16, 13, 15, 11, 220, 17, 13, 23, 11, 220, 18, 13, 15, 11, 220, 19, 13, 15, 11, 220, 20, 13, 15, 11, 220, 17, 13, 15, 1125, 220, 15, 13, 18, 340, 262, 3007, 198, 262, 3190], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [10]:
# Pre process function
# Choose necessary columns for fine tuning and remove the others 
# Format the input for model

# Examples
# The prompt is 


'''
Write some unit tests for this program 

from typing import List

def mean_absolute_deviation(numbers: List[float]) -> float:
""" For a given list of input numbers, calculate Mean Absolute Deviation
around the mean of this dataset.
Mean Absolute Deviation is the average absolute difference between each
element and a centerpoint (mean in this case):
MAD = average | x - x_mean |
>>> mean_absolute_deviation([1.0, 2.0, 3.0, 4.0])
1.0
"""

Test:

mean = sum(numbers) / len(numbers)
return sum(abs(x - mean) for x in numbers) / len(numbers)


'''
# Test:


def tokenization(example):
    start_prompt = "Write some unit tests for this program "
    end_prompt = "\nTest:\n"
    prompt = start_prompt + example['prompt'] + example['canonical_solution'] + end_prompt

    example['input_ids'] = tokenizer(prompt, padding="max_length", truncation=True)['input_ids']
    example['attention_mask'] = tokenizer(prompt, padding="max_length", truncation=True)['attention_mask']
    example['labels'] = tokenizer(example['test'], padding="max_length", truncation=True)['input_ids']
    
    return example

ds = ds.map(tokenization)
ds = ds.remove_columns(['task_id', 'prompt', 'canonical_solution', 'test', 'entry_point'])


Map: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 164/164 [00:05<00:00, 31.24 examples/s]


In [11]:
ds.set_format(type="torch", columns=['input_ids', 'labels'])
ds.format['type']

'torch'

In [12]:


model_name = "Qwen/Qwen2.5-Coder-3B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)


Loading checkpoint shards: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:49<00:00, 24.62s/it]
You are calling `save_pretrained` to a 4-bit converted model, but your `bitsandbytes` version doesn't support it. If you want to save 4-bit models, make sure to have `bitsandbytes>=0.41.3` installed.


In [13]:
model.config.use_cache = False
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

print("load model with bnb config complete")


lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
print("get peft model done")

load model with bnb config complete
get peft model done


In [14]:
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
   logits, labels = eval_pred
   predictions = np.argmax(logits, axis=-1)
   return metric.compute(predictions=predictions, references=labels)



training_args = TrainingArguments(
   output_dir="test_trainer",
   evaluation_strategy="epoch",
   per_device_train_batch_size=1,  # Reduce batch size here
   per_device_eval_batch_size=1,    # Optionally, reduce for evaluation as well
   gradient_accumulation_steps=4,
    num_train_epochs=5,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch"
   )


trainer = Trainer(
   model=model,
   args=training_args,
   train_dataset=ds,
   eval_dataset=ds,

)

device = torch.device('cuda:0')  # Specify GPU device
print(f"Total memory: {torch.cuda.get_device_properties(device).total_memory / 1024**3:.2f} GB")
print(f"Allocated memory: {torch.cuda.memory_allocated(device) / 1024**3:.2f} GB")
print(f"Reserved memory: {torch.cuda.memory_reserved(device) / 1024**3:.2f} GB")

trainer.train()

PATH = "model_old_peft"
trainer.model.save_pretrained(PATH)
tokenizer.save_pretrained(PATH)

# Continute from last checkpoint
#trainer.train(resume_from_checkpoint = True)


/home/vasco/nguyehmk/miniconda3/envs/llm/lib/python3.10/site-packages/accelerate/accelerator.py:432: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches', 'even_batches', 'use_seedable_sampler']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False, even_batches=True, use_seedable_sampler=True)
  warnings.warn(


Total memory: 10.91 GB
Allocated memory: 3.06 GB
Reserved memory: 3.35 GB


OutOfMemoryError: CUDA out of memory. Tried to allocate 4.00 GiB (GPU 0; 10.91 GiB total capacity; 7.31 GiB already allocated; 3.12 GiB free; 7.60 GiB reserved in total by PyTorch) If reserved memory is >> allocated memory try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF

In [ ]:
PATH = "model_old_peft"
trainer.model.save_pretrained(PATH)
tokenizer.save_pretrained(PATH)

In [ ]:
# Load and test the fine_tuned model
model = AutoModelForCausalLM.from_pretrained(PATH, device_map="auto", torch_dtype="auto").to(device)

messages = [
    {"role": "user", "content": "Write some unit test for a function of calculating the factorial of an integer."},
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(device)

outputs = model.generate(**inputs, max_new_tokens=500)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

